## **Aim**
To write a python program for demonstrating Locard exchange principle

## **Algorithm**
**Step 1:** Start

**Step 2:** Define `fmt(ts)` to convert a timestamp into a readable `HH:MM:SS.mmm` string

**Step 3:** Define `snapshot(path)` to capture file size, atime, mtime, ctime, and SHA-256 hash of content

**Step 4:** Define `diff(label, before, after)` to compare two snapshots field-by-field and print what changed

**Step 5:** Create `evidence.txt` with initial content, then take Snapshot 1

**Step 6:** Wait 1 second, read the file, reset its mtime (simulate a read), then take Snapshot 2

**Step 7:** Wait 1 second, append a line to the file (simulate a modification), then take Snapshot 3

**Step 8:** Wait 1 second, rename the file to `evidence_moved.txt`, then take Snapshot 4

**Step 9:** Prepare transition labels: `create -> read`, `read -> modify`, `modify -> rename`

**Step 10:** Pair up consecutive snapshots (1→2, 2→3, 3→4) with their labels

**Step 11:** For each pair, call `diff()` to print which fields changed and which stayed the same

**Step 12:** End

In [1]:
import os
import time
import hashlib
from datetime import datetime


def fmt(ts):
    return datetime.fromtimestamp(ts).strftime("%H:%M:%S.%f")[:-3]


def snapshot(path):
    st = os.stat(path)
    with open(path, "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    return {
        "size": st.st_size,
        "atime": fmt(st.st_atime),
        "mtime": fmt(st.st_mtime),
        "ctime": fmt(st.st_ctime),
        "sha256": digest,
    }


def diff(label, before, after):
    print(f"\n{label}")
    for key in before:
        if before[key] != after[key]:
            print(f"  {key}: {before[key]} -> {after[key]}")
        else:
            print(f"  {key}: {before[key]} -> {after[key]} (unchanged)")


def main():
    path = "evidence.txt"
    open(path, "w").write("original content\n")
    snapshots = [snapshot(path)]

    time.sleep(1)
    open(path).read()
    os.utime(path, (time.time(), os.stat(path).st_mtime))
    snapshots.append(snapshot(path))

    time.sleep(1)
    open(path, "a").write("tampered line\n")
    snapshots.append(snapshot(path))

    time.sleep(1)
    renamed = "evidence_moved.txt"
    os.rename(path, renamed)
    snapshots.append(snapshot(renamed))

    labels = ["create -> read", "read -> modify", "modify -> rename"]
    for label, before, after in zip(labels, snapshots, snapshots[1:]):
        diff(label, before, after)


if __name__ == "__main__":
    main()


create -> read
  size: 17 -> 17 (unchanged)
  atime: 14:26:21.151 -> 14:26:22.152
  mtime: 14:26:21.152 -> 14:26:21.152 (unchanged)
  ctime: 14:26:21.152 -> 14:26:22.152
  sha256: 516ad7b388b21e05e8c56229f063d112e70a2fea45fdd357e8ff44e6a5bce689 -> 516ad7b388b21e05e8c56229f063d112e70a2fea45fdd357e8ff44e6a5bce689 (unchanged)

read -> modify
  size: 17 -> 31
  atime: 14:26:22.152 -> 14:26:22.152 (unchanged)
  mtime: 14:26:21.152 -> 14:26:23.152
  ctime: 14:26:22.152 -> 14:26:23.152
  sha256: 516ad7b388b21e05e8c56229f063d112e70a2fea45fdd357e8ff44e6a5bce689 -> 680e794e736131f2d2dd6ad810d3a493f8bec0fc8d5a639ad70e0733da183ae2

modify -> rename
  size: 31 -> 31 (unchanged)
  atime: 14:26:22.152 -> 14:26:23.153
  mtime: 14:26:23.152 -> 14:26:23.152 (unchanged)
  ctime: 14:26:23.152 -> 14:26:24.152
  sha256: 680e794e736131f2d2dd6ad810d3a493f8bec0fc8d5a639ad70e0733da183ae2 -> 680e794e736131f2d2dd6ad810d3a493f8bec0fc8d5a639ad70e0733da183ae2 (unchanged)


## **Result**
This the program successfully demonstrates the locard's exchange principle